# Local / VS Code Fine-Tuning & Evaluation: Medical Grounded RAG

This notebook is modified to run locally inside **VS Code** (Windows/Linux/Mac) without Google Colab dependencies (`google.colab`, `files.upload`, etc.).

> **Hardware Note:** LLM fine-tuning (e.g., Ministral 3B) is computationally intensive. If running on CPU, you can inspect the data pipeline or test on a micro-sample. For full 4-bit QLoRA training, an NVIDIA GPU with CUDA or Google Colab T4 is recommended.

In [ ]:
# Step 1: Check Environment and Dependencies
import os
import sys
import json
import hashlib
import random
import re
from pathlib import Path

import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("[Notice] Running on CPU. Training will be slow for large models.")

# Locate dataset directory
ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "data").exists() and (ROOT_DIR.parent / "data").exists():
    ROOT_DIR = ROOT_DIR.parent

MANIFEST_PATH = ROOT_DIR / "data" / "processed" / "synthea_manifest.jsonl"
TRAIN_SFT_PATH = ROOT_DIR / "data" / "processed" / "train_sft.jsonl"
VAL_SFT_PATH = ROOT_DIR / "data" / "processed" / "val_sft.jsonl"

print(f"Manifest Path: {MANIFEST_PATH.resolve()}")
print(f"Manifest exists: {MANIFEST_PATH.exists()}")

In [ ]:
# Step 2: Load and Prepare Training & Evaluation Data
SYSTEM_PROMPT = (
    "You are a strict medical-record assistant. Answer only from the supplied record excerpt. "
    "If the excerpt does not support the answer, respond exactly: "
    "Information not found in medical records."
)
ABSTAIN = "Information not found in medical records."

def extract_example(record):
    lines = [x.strip() for x in record.get('text', '').splitlines() if x.strip()]
    candidates = lines[1:] or lines
    if not candidates:
        return None
    
    hash_val = int(hashlib.sha256(record['patient_id'].encode()).hexdigest(), 16)
    excerpt = candidates[hash_val % len(candidates)][:600]
    answerable = (hash_val % 2 == 0)
    question = "What is explicitly documented in this medical-record excerpt?" if answerable else "What is the patient's blood type and Rh factor?"
    answer = f"The record states: {excerpt}" if answerable else ABSTAIN
    
    return {
        "patient_id": record['patient_id'],
        "user": f"PATIENT RECORD EXCERPT:\n{excerpt}\n\nQUESTION: {question}",
        "answer": answer,
        "answerable": answerable
    }

if MANIFEST_PATH.exists():
    records = [json.loads(line) for line in MANIFEST_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
    train_data = [extract_example(r) for r in records if r.get('split') == 'train']
    test_data = [extract_example(r) for r in records if r.get('split') == 'test']
    train_data = [x for x in train_data if x is not None]
    test_data = [x for x in test_data if x is not None]
    print(f"Loaded {len(train_data)} train samples and {len(test_data)} held-out test samples from manifest.")
else:
    print("[!] synthea_manifest.jsonl not found. Run 'python prepare_synthea.py' first.")

In [ ]:
# Step 3: Inspect Data Sample
if 'train_data' in locals() and train_data:
    sample = train_data[0]
    print("--- SAMPLE TRAINING ITEM ---")
    print(f"Patient ID: {sample['patient_id']}")
    print(f"User Prompt:\n{sample['user']}")
    print(f"\nTarget Output:\n{sample['answer']}")

In [ ]:
# Step 4: Fine-Tuning Setup with Transformers & PEFT
# Install required packages if missing: pip install transformers peft trl datasets accelerate
try:
    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
    from peft import LoraConfig, get_peft_model
    from trl import SFTTrainer
    
    MODEL_ID = "mistralai/Ministral-3-3B-Instruct-2512"  # Note: Requires Hugging Face login
    print(f"Target Model: {MODEL_ID}")
    print("Hugging Face & Fine-Tuning libraries successfully imported.")
except ImportError as e:
    print(f"[!] Required libraries missing for training: {e}")
    print("Run: pip install transformers peft trl datasets accelerate")